# NoC Architecture Generation - Model Validation on Google Colab

This notebook validates the trained Mistral-7B model outputs using the constraint validation module.

**Setup Instructions:**
1. Upload this notebook to Google Colab
2. Enable GPU: Runtime → Change runtime type → T4 GPU
3. Upload your trained model or mount Google Drive
4. Run all cells to validate model outputs

## 1. Check GPU Availability

In [ ]:
# Check CUDA/GPU availability
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 2. Install Dependencies

In [ ]:
# Install required packages
!pip install -q transformers datasets peft trl bitsandbytes accelerate sentencepiece protobuf

## 3. Upload Project Files

Upload these files from your local project:
- `src/validate_architecture.py` - Validation module
- `data/processed_str/valid.jsonl` - Test data
- Trained model files (or download from Google Drive)

In [ ]:
# Create directory structure
!mkdir -p src data/processed_str outputs/mistral7b-noc-switch-qlora

In [ ]:
# Option: Mount Google Drive to access files
from google.colab import drive
drive.mount('/content/drive')

# Copy files from Drive (adjust paths as needed)
# !cp -r /content/drive/MyDrive/CaseStudy-ezhil/src/validate_architecture.py src/
# !cp -r /content/drive/MyDrive/CaseStudy-ezhil/data/processed_str/valid.jsonl data/processed_str/
# !cp -r /content/drive/MyDrive/CaseStudy-ezhil/outputs/mistral7b-noc-switch-qlora/* outputs/mistral7b-noc-switch-qlora/

## 4. Create Validation Module

In [ ]:
%%writefile src/validate_architecture.py
#!/usr/bin/env python3
"""NoC Architecture Validation Module"""
import json
from typing import Dict, List, Tuple

class ArchitectureValidator:
    """Validates NoC architecture constraints."""
    
    def __init__(self, spec: Dict, output: Dict):
        self.spec = spec
        self.output = output
        self.errors = []
        self.warnings = []
    
    def validate_all(self) -> Tuple[bool, Dict]:
        self.errors = []
        self.warnings = []
        
        self._validate_switch_placement()
        self._validate_path_elements()
        self._validate_route_connectivity()
        self._validate_no_cycles()
        
        is_valid = len(self.errors) == 0
        
        report = {
            "valid": is_valid,
            "errors": self.errors,
            "warnings": self.warnings,
            "checks": {
                "switch_placement": not self._has_error("switch_placement"),
                "path_elements": not self._has_error("path_elements"),
                "route_connectivity": not self._has_error("route_connectivity"),
                "no_cycles": not self._has_error("cycles")
            }
        }
        
        return is_valid, report
    
    def _has_error(self, error_type: str) -> bool:
        return any(error_type in e for e in self.errors)
    
    def _validate_switch_placement(self):
        floorplan = self.spec.get("floorplan_dim", [1000, 1000])
        max_x, max_y = floorplan[0], floorplan[1]
        blockages = self.spec.get("blockages", {})
        switches = self.output.get("switches", {})
        
        for switch_id, coords in switches.items():
            x, y = coords["x"], coords["y"]
            
            if not (0 <= x <= max_x and 0 <= y <= max_y):
                self.errors.append(
                    f"switch_placement: {switch_id} at ({x}, {y}) outside bounds ({max_x}, {max_y})"
                )
            
            for block_id, block in blockages.items():
                bx, by = block["x"], block["y"]
                bw, bh = block["width"], block["height"]
                
                if (bx <= x <= bx + bw) and (by <= y <= by + bh):
                    self.errors.append(
                        f"switch_placement: {switch_id} at ({x}, {y}) inside blockage {block_id}"
                    )
    
    def _validate_path_elements(self):
        inits = set(self.spec.get("inits", {}).keys())
        targets = set(self.spec.get("targets", {}).keys())
        switches = set(self.output.get("switches", {}).keys())
        
        all_valid_nodes = inits | targets | switches
        routing_paths = self.output.get("routing_paths", {})
        
        for route_id, path in routing_paths.items():
            if not isinstance(path, list):
                self.errors.append(f"path_elements: {route_id} path is not a list")
                continue
            
            for node in path:
                if node not in all_valid_nodes:
                    self.errors.append(
                        f"path_elements: {route_id} contains non-existent node '{node}'"
                    )
    
    def _validate_route_connectivity(self):
        connectivity = self.spec.get("connectivity", {})
        routing_paths = self.output.get("routing_paths", {})
        
        for route_id, (init, target) in connectivity.items():
            if route_id not in routing_paths:
                self.errors.append(
                    f"route_connectivity: Required route {route_id} ({init}->{target}) is missing"
                )
                continue
            
            path = routing_paths[route_id]
            
            if len(path) < 2:
                self.errors.append(
                    f"route_connectivity: {route_id} path too short"
                )
            elif path[0] != init:
                self.errors.append(
                    f"route_connectivity: {route_id} should start with {init}, got {path[0]}"
                )
            elif path[-1] != target:
                self.errors.append(
                    f"route_connectivity: {route_id} should end with {target}, got {path[-1]}"
                )
    
    def _validate_no_cycles(self):
        routing_paths = self.output.get("routing_paths", {})
        
        for route_id, path in routing_paths.items():
            if not isinstance(path, list) or len(path) < 2:
                continue
            
            seen_nodes = set()
            for node in path:
                if node in seen_nodes:
                    self.errors.append(
                        f"cycles: Route {route_id} contains loop - '{node}' appears multiple times"
                    )
                    break
                seen_nodes.add(node)

def validate_architecture(spec: Dict, output: Dict) -> Tuple[bool, Dict]:
    validator = ArchitectureValidator(spec, output)
    return validator.validate_all()

## 5. Load Test Data

In [ ]:
import json

# Load validation dataset
test_samples = []
with open('data/processed_str/valid.jsonl', 'r') as f:
    for line in f:
        test_samples.append(json.loads(line))

print(f"Loaded {len(test_samples)} test samples")

# Show first sample structure
print("\nSample structure:")
print(f"Keys: {test_samples[0].keys()}")
print(f"Prompt length: {len(test_samples[0]['prompt'])} chars")
print(f"Label length: {len(test_samples[0]['label'])} chars")

## 6. Load Trained Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

# Configuration
base_model_name = "mistralai/Mistral-7B-Instruct-v0.2"
adapter_path = "outputs/mistral7b-noc-switch-qlora"

# QLoRA config for inference
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(model, adapter_path)

print("\n✅ Model loaded successfully!")
print(f"Device: {next(model.parameters()).device}")

## 7. Generate and Validate Outputs

In [ ]:
from src.validate_architecture import validate_architecture
import re

def extract_output_json(generated_text, prompt):
    """Extract JSON output from generated text."""
    # Remove the prompt
    response = generated_text[len(prompt):].strip()
    
    # Try to find JSON in the response
    json_match = re.search(r'\{.*\}', response, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group())
        except:
            pass
    return None

def generate_and_validate(sample, model, tokenizer, max_new_tokens=512):
    """Generate output and validate it."""
    prompt = sample['prompt']
    spec = json.loads(sample['prompt'].split('Specification:\n')[1].split('\n\nGenerate')[0])
    
    # Generate
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract output JSON
    output = extract_output_json(generated_text, prompt)
    
    if output is None:
        return {
            'valid': False,
            'errors': ['Failed to extract valid JSON from output'],
            'output': None,
            'generated_text': generated_text
        }
    
    # Validate
    is_valid, report = validate_architecture(spec, output)
    
    return {
        'valid': is_valid,
        'errors': report['errors'],
        'checks': report['checks'],
        'output': output,
        'generated_text': generated_text
    }

print("Ready to generate and validate!")

## 8. Run Validation on Sample

In [ ]:
# Test on first sample
print("Generating and validating sample 0...\n")

result = generate_and_validate(test_samples[0], model, tokenizer)

print("="*60)
print("VALIDATION RESULT")
print("="*60)
print(f"Valid: {result['valid']}")
print(f"\nChecks:")
for check_name, passed in result.get('checks', {}).items():
    status = "✅" if passed else "❌"
    print(f"  {status} {check_name}")

if result['errors']:
    print(f"\nErrors ({len(result['errors'])}):")
    for error in result['errors'][:5]:
        print(f"  - {error}")

if result['output']:
    print(f"\nGenerated Output:")
    print(f"  Switches: {len(result['output'].get('switches', {}))}")
    print(f"  Routing Paths: {len(result['output'].get('routing_paths', {}))}")
else:
    print("\nFailed to parse output as JSON")
    print(f"Generated text preview:")
    print(result['generated_text'][:500])

## 9. Batch Validation

In [ ]:
# Validate multiple samples
num_samples = min(10, len(test_samples))  # Test first 10 samples

print(f"Validating {num_samples} samples...\n")

results = []
for i in range(num_samples):
    print(f"Sample {i+1}/{num_samples}...", end=" ")
    result = generate_and_validate(test_samples[i], model, tokenizer)
    results.append(result)
    print("✅ Valid" if result['valid'] else "❌ Invalid")

# Calculate metrics
valid_count = sum(1 for r in results if r['valid'])
invalid_count = len(results) - valid_count
validity_rate = 100 * valid_count / len(results)

print("\n" + "="*60)
print("VALIDATION SUMMARY")
print("="*60)
print(f"Total Samples: {len(results)}")
print(f"Valid: {valid_count} ({validity_rate:.1f}%)")
print(f"Invalid: {invalid_count}")

# Error analysis
all_errors = []
for r in results:
    all_errors.extend(r['errors'])

if all_errors:
    print(f"\nTotal Errors: {len(all_errors)}")
    print("\nMost Common Errors:")
    error_types = {}
    for error in all_errors:
        error_type = error.split(':')[0]
        error_types[error_type] = error_types.get(error_type, 0) + 1
    
    for error_type, count in sorted(error_types.items(), key=lambda x: x[1], reverse=True):
        print(f"  {error_type}: {count}")

## 10. Detailed Analysis

In [ ]:
# Analyze validation checks
check_stats = {
    'switch_placement': 0,
    'path_elements': 0,
    'route_connectivity': 0,
    'no_cycles': 0
}

for result in results:
    for check_name, passed in result.get('checks', {}).items():
        if passed:
            check_stats[check_name] += 1

print("Constraint Pass Rates:")
for check_name, count in check_stats.items():
    rate = 100 * count / len(results)
    print(f"  {check_name}: {count}/{len(results)} ({rate:.1f}%)")

## 11. Visualize Results

In [ ]:
import matplotlib.pyplot as plt

# Validity pie chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Validity distribution
ax1.pie(
    [valid_count, invalid_count],
    labels=['Valid', 'Invalid'],
    autopct='%1.1f%%',
    colors=['#2ecc71', '#e74c3c'],
    startangle=90
)
ax1.set_title('Architecture Validity Distribution')

# Constraint pass rates
checks = list(check_stats.keys())
pass_rates = [100 * check_stats[c] / len(results) for c in checks]

ax2.barh(checks, pass_rates, color='#3498db')
ax2.set_xlabel('Pass Rate (%)')
ax2.set_title('Constraint Validation Pass Rates')
ax2.set_xlim(0, 100)
ax2.grid(axis='x', alpha=0.3)

for i, v in enumerate(pass_rates):
    ax2.text(v + 1, i, f'{v:.1f}%', va='center')

plt.tight_layout()
plt.show()

## 12. Compare with Ground Truth

In [ ]:
# Compare generated outputs with ground truth
print("Comparing with Ground Truth:\n")

for i in range(min(3, len(results))):
    sample = test_samples[i]
    result = results[i]
    
    # Extract ground truth from label
    gt_output = json.loads(sample['label'])
    gen_output = result.get('output', {})
    
    print(f"Sample {i}:")
    print(f"  Ground Truth - Switches: {len(gt_output.get('switches', {}))}")
    print(f"  Generated    - Switches: {len(gen_output.get('switches', {}))}")
    print(f"  Ground Truth - Routes: {len(gt_output.get('routing_paths', {}))}")
    print(f"  Generated    - Routes: {len(gen_output.get('routing_paths', {}))}")
    print(f"  Valid: {result['valid']}\n")

## 13. Save Validation Report

In [ ]:
# Save detailed report
report = {
    'summary': {
        'total_samples': len(results),
        'valid_count': valid_count,
        'invalid_count': invalid_count,
        'validity_rate': validity_rate
    },
    'constraint_pass_rates': {
        check: 100 * count / len(results)
        for check, count in check_stats.items()
    },
    'error_analysis': error_types if all_errors else {},
    'samples': [
        {
            'index': i,
            'valid': r['valid'],
            'errors': r['errors'],
            'checks': r.get('checks', {})
        }
        for i, r in enumerate(results)
    ]
}

with open('validation_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print("✅ Validation report saved to validation_report.json")

# Download the report
from google.colab import files
files.download('validation_report.json')

## Summary

This notebook validates trained NoC generation model outputs using:
- ✅ GPU acceleration (CUDA)
- ✅ 4-bit quantized model loading
- ✅ Constraint-based validation
- ✅ Comprehensive error analysis
- ✅ Visual metrics and reporting

**Key Metrics:**
- Validity Rate: Percentage of architectures passing all constraints
- Constraint Pass Rates: Per-constraint success rates
- Error Analysis: Most common validation failures

The validation report can be used to:
1. Assess model quality
2. Identify common failure modes
3. Guide further training or fine-tuning
4. Compare different model checkpoints